# Equilibrium Matching on 2D Toy Datasets

This notebook implements **Equilibrium Matching** (Wang & Du, 2025).

We test on 2D toy distributions and investigate how the choice of decay schedule and
descent method affects sample quality.

## Setup — imports and seeds

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from tqdm.auto import tqdm
import ot  

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# seeds 
torch.manual_seed(42)
np.random.seed(42)

## Target datasets



In [ ]:
def sample_moons(n, noise=0.05):
    """Sample n points from the two-moons distribution."""
    X, _ = make_moons(n_samples=n, noise=noise)
    return torch.tensor(X, dtype=torch.float32)


def sample_gaussian_mixture(n, n_components=8, radius=3.0, std=0.3):
    """Sample n points from a ring of Gaussian blobs."""
    # place cluster centres
    angles = torch.linspace(0, 2 * np.pi, n_components + 1)[:-1]
    means = radius * torch.stack([torch.cos(angles), torch.sin(angles)], dim=1)
    # randomly assign each point to a cluster, then add Gaussian noise
    idx = torch.randint(0, n_components, (n,))
    return means[idx] + std * torch.randn(n, 2)


def sample_checkerboard(n, size=4):
    """Sample n points from a checkerboard pattern."""
    samples = []
    while len(samples) < n:
        # generate random points in a grid
        x = torch.rand(n * 4, 2) * size - size / 2
        ix = torch.floor(x[:, 0]).long()
        iy = torch.floor(x[:, 1]).long()
        samples.append(x[(ix + iy) % 2 == 0])
    return torch.cat(samples)[:n]

## Gradient network

The network takes just a 2D point and outputs a 2D gradient vector. 

In [ ]:
class GradientMLP(nn.Module):

    def __init__(self, data_dim=2, hidden_dim=256, n_layers=4):
        super().__init__()
        layers = [nn.Linear(data_dim, hidden_dim), nn.SiLU()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.SiLU()]
        layers.append(nn.Linear(hidden_dim, data_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

## Decay functions

The decay function $c(\gamma)$ controls how strongly the training signal is weighted at
different interpolation levels $\gamma \in [0, 1]$. Near $\gamma = 1$ (close to data),
the decay should vanish so the gradient learns to be zero at the data points.

We compare three schedules:
- **Linear**: $c(\gamma) = \lambda(1 - \gamma)$ 
- **Truncated**: flat at $\lambda$ until $\gamma = a$, then linear decay to zero
- **Piecewise**: starts above $\lambda$, linearly decreases, with a steeper drop after $a$

In [ ]:
def c_linear(gamma, lam=1.0):
    return (lam * (1 - gamma)).unsqueeze(-1)


def c_truncated(gamma, a=0.8, lam=4.0):
    c = torch.where(gamma <= a, torch.ones_like(gamma), (1 - gamma) / (1 - a))
    return (lam * c).unsqueeze(-1)


def c_piecewise(gamma, a=0.8, b=1.4, lam=4.0):
    high = b - (b - 1) / a * gamma          
    low = (1 - gamma) / (1 - a)             
    c = torch.where(gamma <= a, high, low)
    return (lam * c).unsqueeze(-1)

## Training loop


In [ ]:
def train_eqm(model, sample_data_fn, n_steps=10_000, batch_size=512, lr=1e-3, decay_fn=c_truncated):
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []

    pbar = tqdm(range(n_steps), desc="Training")
    for step in pbar:
        # x1 = real data, x0 = noise
        x1 = sample_data_fn(batch_size).to(device)
        x0 = torch.randn_like(x1)

        # random interpolation level gamma in [0, 1]
        gamma = torch.rand(batch_size, device=device)

        # interpolate
        g_ = gamma.unsqueeze(-1)
        xg = (1 - g_) * x0 + g_ * x1

        target = (x0 - x1) * decay_fn(gamma)

        # train the network to predict this gradient
        pred = model(xg)
        loss = (pred - target).pow(2).mean()

        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        losses.append(loss.item())
        if step % 500 == 0:
            pbar.set_postfix(loss=f"{loss.item():.4f}")

    return losses

## Generation (iterative descent)

To generate samples, we start from noise and repeatedly step in the negative gradient
direction. We implement four descent methods:
- **GD** — plain gradient descent
- **Heavy Ball** — Polyaks Heavy Ball
- **Nesterov** — Nesterov Accelerated Gradient
- **Langevin** — GD plus random noise

In [ ]:
@torch.no_grad()
def generate(model, n_samples=1000, n_steps=250, eta=0.02,
             method="gd", mu=0.35, T=0.1, eps=None,
             return_trajectories=False):
    model.eval()
    x = torch.randn(n_samples, 2, device=device)  # start from pure noise
    x_last = x.clone()  # needed for momentum methods
    converged = torch.zeros(n_samples, dtype=torch.bool, device=device)

    if return_trajectories:
        trajectory = [x.cpu().clone()]

    for i in range(n_steps):
        grad = model(x)

        # optional early stopping
        if eps is not None:
            converged = converged | (grad.norm(dim=-1) < eps)

        if method == "gd":
            # gradient descent
            update = -eta * grad

        elif method == "nesterov":
            x_look = x + mu * (x - x_last)
            grad = model(x_look)
            update = -eta * grad
            x_last = x.clone()

        elif method == "langevin":
            # controlled by temperature T
            noise = torch.randn_like(x)
            update = -eta * grad + (2 * eta * T) ** 0.5 * noise

        elif method == "heavyball":
            update = -eta * grad + mu * (x - x_last)
            x_last = x.clone()

        if eps is not None:
            update[converged] = 0.0

        x = x + update

        if return_trajectories:
            trajectory.append(x.cpu().clone())

    model.train()
    if return_trajectories:
        return x.cpu(), torch.stack(trajectory, dim=1)
    return x.cpu()

## Evaluation metric

We measure sample quality using the **Wasserstein-2** ($W_2$) distance between generated
and true samples. Lower = better. 

In [ ]:
def compute_w2(generated, reference):
    """Wasserstein-2 distance."""
    n, m = len(generated), len(reference)
    a = np.ones(n) / n
    b = np.ones(m) / m
    C = ot.dist(generated, reference, metric="sqeuclidean")
    return np.sqrt(ot.emd2(a, b, C))

## Plotting helpers

In [ ]:
def plot_progression(trajectories, steps_to_show, xlim, ylim):
    """Show the sample distribution at specific descent steps."""
    fig, axes = plt.subplots(1, len(steps_to_show), figsize=(4.0 * len(steps_to_show), 4.0), facecolor="white")
    for ax, step_idx in zip(np.atleast_1d(axes), steps_to_show):
        pts = trajectories[:, step_idx].numpy()
        ax.scatter(pts[:, 0], pts[:, 1], s=2.0, alpha=0.4, linewidths=0)
        ax.set_title(f"step {step_idx}", fontsize=12)
        ax.set_xlim(*xlim); ax.set_ylim(*ylim)
        ax.set_aspect("equal")
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
    plt.tight_layout()
    plt.show()


def plot_field_magnitude(model, xlim, ylim, resolution=500):
    """Heatmap of ||f(x)|| over the plane."""
    xs = torch.linspace(xlim[0], xlim[1], resolution)
    ys = torch.linspace(ylim[0], ylim[1], resolution)
    xx, yy = torch.meshgrid(xs, ys, indexing="xy")
    grid = torch.stack([xx.flatten(), yy.flatten()], dim=1).to(device)

    model.eval()
    with torch.no_grad():
        magnitude = model(grid).norm(dim=-1).cpu().reshape(resolution, resolution).numpy()
    model.train()

    fig, ax = plt.subplots(figsize=(6, 6), facecolor="white")
    ax.imshow(magnitude, extent=[*xlim, *ylim], origin="lower", aspect="equal", interpolation="bilinear")
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    plt.tight_layout()
    plt.show()


def plot_descent_comparison(model, methods, step_counts, n_samples=4000, eta=0.005,
                            xlim=(-4.5, 4.5), ylim=(-4.5, 4.5)):
    
    colors = {"GD": "#0072B2", "Heavy Ball": "#D55E00", "Nesterov": "#009E73", "Langevin": "#CC79A7"}
    fig, axes = plt.subplots(len(step_counts), len(methods),
                             figsize=(4 * len(methods), 4 * len(step_counts)), facecolor="white")
    for row, n_steps in enumerate(step_counts):
        for col, (name, kwargs) in enumerate(methods.items()):
            ax = axes[row, col]
            torch.manual_seed(0)  # same noise for fair comparison
            samples = generate(model, n_samples=n_samples, n_steps=n_steps, eta=eta, **kwargs)
            pts = samples.numpy()
            ax.scatter(pts[:, 0], pts[:, 1], s=2, alpha=0.4, linewidths=0, color=colors[name])
            ax.set_xlim(*xlim); ax.set_ylim(*ylim)
            ax.set_aspect("equal")
            ax.set_xticks([]); ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)
            if row == 0:
                ax.set_title(name, fontsize=13)
            if col == 0:
                ax.set_ylabel(f"{n_steps} steps", fontsize=12, rotation=90, labelpad=10)
    plt.tight_layout()
    plt.show()

## Experiment 1 — EqM progression

Train on **8 Gaussians** and **Two Moons** (15k steps, truncated decay with $a=0.8$,
$\lambda=4$), then show how the samples evolve over gradient descent steps.

In [ ]:
# train on the 8-Gaussians dataset
model_8g = GradientMLP().to(device)
losses_8g = train_eqm(
    model_8g,
    sample_data_fn=lambda n: sample_gaussian_mixture(n),
    n_steps=15_000,
)

In [ ]:
# generate with GD (eta=0.005) and show snapshots at steps 0, 25, 50, 100, 250
_, traj_8g = generate(model_8g, n_samples=4000, n_steps=250, eta=0.005, return_trajectories=True)
plot_progression(traj_8g, [0, 25, 50, 100, 250], xlim=(-4.5, 4.5), ylim=(-4.5, 4.5))

In [ ]:
# train on the two-moons dataset
model_moons = GradientMLP().to(device)
losses_moons = train_eqm(
    model_moons,
    sample_data_fn=lambda n: sample_moons(n, noise=0.05),
    n_steps=15_000,
)

In [ ]:
# same thing for two moons
_, traj_moons = generate(model_moons, n_samples=4000, n_steps=250, eta=0.005, return_trajectories=True)
plot_progression(traj_moons, [0, 25, 50, 100, 250], xlim=(-3.0, 3.5), ylim=(-2.0, 2.5))

## Experiment 2 — Learned field magnitude

Visualise $\|f_\theta(x)\|$ over the plane.

In [ ]:
# gradient magnitude for the 8-Gaussians model
plot_field_magnitude(model_8g, xlim=(-5, 5), ylim=(-5, 5))

In [ ]:
# gradient magnitude for the two-moons model
plot_field_magnitude(model_moons, xlim=(-3, 3.5), ylim=(-3, 3.5))

## Experiment 3 — Comparison of descent methods

Compare four ways to descend the gradient field on the 8-Gaussians model:
GD, Heavy Ball ($\mu=0.35$), Nesterov ($\mu=0.35$), and Langevin ($T=0.01$).
We show results at 50, 100, and 250 steps to see how each converges.

In [ ]:

methods = {
    "GD":         dict(method="gd"),
    "Heavy Ball": dict(method="heavyball", mu=0.35),
    "Nesterov":   dict(method="nesterov",  mu=0.35),
    "Langevin":   dict(method="langevin",  T=0.01),
}

plot_descent_comparison(model_8g, methods, step_counts=[50, 100, 250])

## Experiment 4 — Comparison of decay schedules

Train three separate models on **Two Moons** (10k steps each) using different decay
functions (all with $\lambda=4$). Then measure $W_2$ at various step counts.

In [ ]:
sample_fn = lambda n: sample_moons(n, noise=0.05)

decays = {
    "Linear":    lambda g: c_linear(g, lam=4.0),
    "Truncated": lambda g: c_truncated(g, a=0.8, lam=4.0),
    "Piecewise": lambda g: c_piecewise(g, a=0.8, b=1.4, lam=4.0),
}

models_decay = {}
for name, decay_fn in decays.items():
    torch.manual_seed(0)
    np.random.seed(0)
    m = GradientMLP().to(device)
    losses = train_eqm(m, sample_data_fn=sample_fn, n_steps=10_000, decay_fn=decay_fn)
    models_decay[name] = m

In [ ]:
n_eval = 2000
eta = 0.005
step_counts = [0, 10, 25, 50, 100, 250, 500]

# baseline: W2 between two independent sets of true samples
torch.manual_seed(123)
reference = sample_fn(n_eval).numpy()

torch.manual_seed(456)
ref2 = sample_fn(n_eval).numpy()
print(f"Baseline W2 (true vs true): {compute_w2(ref2, reference):.4f}")

# evaluate each decay schedule at every step count
results = {name: [] for name in models_decay}
for name, m in models_decay.items():
    for k in step_counts:
        torch.manual_seed(0)  # same starting noise
        samples = generate(m, n_samples=n_eval, n_steps=k, eta=eta, method="gd").numpy()
        w2 = compute_w2(samples, reference)
        results[name].append(w2)
        print(f"{name:>12s}  steps={k:>4d}  W2={w2:.4f}")